# GISLR — Fewer missed signs than the confidence floor, at no more wrong or extra signs

**What this notebook does.** It is a diagnostic experiment on stage 2 of sign → speech (TODO §12.6,
floor-recall experiment, approved 2026-09-24), and no model runs here. It reuses C1's cached
per-frame outputs from `gislr.4.downstream.next-gloss.ipynb` §3.

**The problem.** The confidence floor (accept a segment only if `q_max ≥ 0.3`) is what
makes noisy streams usable: GER 0.982 → 0.580, and noise spoken 75% → 39%. On clean
sentences, though, it turns wrong signs into missed ones: substitutions 0.213 → 0.091, but
deletions 0.064 → 0.252. A diagnostic on the selection signers found:
- 32% of the clean segments it rejects were **correct**;
- the true sign of a rejected segment is in `q`'s top 5 60% of the time;
- one threshold rejects both noise and uncertain real signs.

**The question.** Can we keep missed signs lower **at the floor's own wrong-sign and extra-sign
rates** (and its noise false-accept rate)?

**Arms** (`sb.recognize.continuous.select`, TODO §12.6 plan):

| § | arm | idea |
|---|---|---|
| 2 | `scores` | Another score for the same one-segment decision: `peak` (the gloss's best single frame), `margin` (top-1 minus top-2), `qp` (√(q·peak)); `q` itself is the floor. Each without the prior and with it (λ=0.3) |
| 3 | `class` | Per-sign floors: the floor shifted by how often segments voted as that gloss are right on the selection signers, shrunk toward the average |
| 4 | `lattice` | **Wait before deciding.** Hold a segment until `lag` more have closed, then take the best path over each held segment's top-k glosses or "skip", with the trigram on both sides |
| 5 | `linear` | A logistic regression on 10 segment and prior features, trained on selection signers' clean and noisy streams together, so it can tell noise from uncertainty. **This cell fits a model** (seconds, CPU) |

**Protocol:** the same as the next-gloss notebook. Settings are chosen on the 5 selection
signers and reported on the other 16.
- **Budget matching (§6):** a setting is eligible only if, on the selection signers' clean
  *and* noisy sentences, its substitution and insertion rates are each at most the floor's
  + 0.002, and its noise false-accept rate is at most the floor's + 0.005. Among eligible
  settings, the lowest deletion rate wins.
- **Reporting:** on the evaluation signers, with Δ vs the floor and 95% confidence
  intervals from a per-signer bootstrap.

**Artifacts** (`data/cache/gislr/downstream/acceptance/`):

| path | what |
|---|---|
| `results/labels_select.parquet` | every selection-signer D3 segment, aligned: match / sub / ins, true gloss, rank of the true gloss in `q` |
| `results/refs_select.json` | baseline, floor, floor+prior on the selection signers (the budget) |
| `results/sweep_<arm>.json` | selection sweeps (resumable per setting) |
| `results/class_shifts.npy`, `results/linear_lam<λ>.json` | fitted per-sign shifts, logistic-regression weights (plain numbers: the client's format) |
| `results/final.json`, `results/final_table.csv`, `results/deltas.csv`, `results/per_stream/*.parquet` | evaluation-signer results, bootstrap Δ vs the floor |
| `assets/*.png` | trade-off curves, error breakdown |

**Resumable:** each cell skips any file that already exists, and sweeps resume setting by
setting. `LIMIT_PER_GROUP` runs a smoke test into `acceptance_smoke/`.

**Runtime:** about 10–20 min in total on CPU (a 30-stream smoke run took 21 s). The lattice
sweep (§4) is the longest cell.

**Caveats:**
- Training labels come from aligning *every* segment's argmax against the reference. Every
  reported number comes from real decoding.
- The linear selector's prior features are computed with history = all previous segments
  accepted, which is close to, but not the same as, the history it sees when decoding.
- The noise is synthetic (see the next-gloss notebook).

## Setup

In [ ]:
# ============================================================
# Imports, configs, signer split, corpus folds, trigram prior, cached streams
# ============================================================
import functools
import itertools
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_sentences_dir
from sb.recognize import streaming as S
from sb.recognize.continuous import cache as FC
from sb.recognize.continuous import fuse as FU
from sb.recognize.continuous import select as SL
from sb.recognize.sequences import baselines as B
from sb.recognize.sequences import metrics as M
from sb.recognize.sequences.corpus import load_sentences
from sb.rescore import prior as P

CFG_DIR = EXPERIMENTS_DIR / "recognition" / "configs"
CONFIG = json.loads((CFG_DIR / "gislr.acceptance.json").read_text(encoding="utf-8"))
DS = json.loads((CFG_DIR / CONFIG["downstream_config"]).read_text(encoding="utf-8"))

LIMIT_PER_GROUP = None  # e.g. 30 -> smoke run on the first 30 streams per variant/group, into acceptance_smoke/

DATASET_ROOT = gislr_sentences_dir(DS["dataset_version"])
FWD = CACHE_DIR / CONFIG["forward_dir"]
OUT = CACHE_DIR / "gislr" / "downstream" / ("acceptance" if LIMIT_PER_GROUP is None else "acceptance_smoke")
RES, ASSETS, PER_STREAM = OUT / "results", OUT / "assets", OUT / "results" / "per_stream"
for d in (RES, ASSETS, PER_STREAM):
    d.mkdir(parents=True, exist_ok=True)
assert (FWD / "sentence-noisy" / "select").exists(), "run gislr.4.downstream.next-gloss.ipynb §3 first"


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


# ---- streams, signers, labels: identical to the next-gloss notebook
SEQ_FULL = pd.read_csv(DATASET_ROOT / "sequences.csv", keep_default_na=False)
PARTICIPANT = SEQ_FULL["participant_id"].to_numpy()
SENT_ID, KIND = SEQ_FULL["sentence_id"].to_numpy(), SEQ_FULL["kind"].to_numpy()
SELECT, EVAL = B.signer_split(SEQ_FULL["participant_id"], DS["n_selection_signers"], DS["seed"])
LABEL_MAP = json.loads((DATASET_ROOT / "sign_to_prediction_index_map.json").read_text(encoding="utf-8"))
IDX2GLOSS = [g for g, _ in sorted(LABEL_MAP.items(), key=lambda kv: kv[1])]
N_CLASSES = NULL = len(IDX2GLOSS)  # C1's null index is the last class (asserted in the next-gloss notebook)

# ---- D3 settings chosen by the continuous eval (collapsed repeats)
CEVAL = json.loads((CACHE_DIR / DS["continuous_eval_results"] / "final.json").read_text())
ch = pd.DataFrame(CEVAL["chosen"])
ch = ch[(ch.run == DS["run"]) & (ch.collapse == DS["collapse"]) & (ch.decoder == "D3")].iloc[0]
NU, MIN_LEN = float(ch["nu"]), int(ch["min_len"])

# ---- the trigram prior, fitted without the stream's own sentence (control: full corpus)
SENTS = load_sentences(DS["corpus_version"])
GLOSSES = sorted({g for s in SENTS for g in s.glosses})
FAMILY, ARM = CONFIG["prior"]["family"], CONFIG["prior"]["arm"]
ids, themes = [s.id for s in SENTS], [s.theme for s in SENTS]
FOLDS = {"sentence-fold": P.sentence_folds(ids, themes, DS["lm"]["n_folds"], DS["seed"]),
         "theme-out": P.theme_folds(ids, themes)}[FAMILY]
PERM = np.array([GLOSSES.index(g) for g in IDX2GLOSS])


@functools.cache
def _prior_fold(fold: int):
    train = [s.glosses for s in SENTS if fold == -1 or FOLDS[s.id] != fold]
    lm, memo = P.NgramLM(GLOSSES, DS["lm"]["ngram_orders"][ARM], DS["lm"]["discount"]).fit(train), {}

    def prior(hist, lm=lm, memo=memo):
        if hist not in memo:
            memo[hist] = lm.gloss_dist([IDX2GLOSS[c] for c in hist])[PERM]
        return memo[hist]
    return prior


def prior_for(row: int):
    return _prior_fold(FOLDS.get(SENT_ID[row], -1) if KIND[row] == "sentence" else -1)


@functools.cache
def streams(variant: str, group: str) -> tuple:
    """Cached streams of one variant/group with their D3 segments (per-frame arrays dropped)."""
    out = []
    for st in FC.iter_chunks(FWD / variant / group):
        st["segs"] = FU.segments_d3(st["gp"], NU, MIN_LEN, NULL)
        del st["gp"], st["bp"]
        out.append(st)
        if LIMIT_PER_GROUP is not None and len(out) >= LIMIT_PER_GROUP:
            break
    return tuple(out)


# ---- rules from their dict form (sweep rows, config)
def load_shifts():
    return np.load(RES / "class_shifts.npy")


def load_linear(lam: float) -> SL.LogReg:
    d = json.loads((RES / f"linear_lam{lam}.json").read_text())
    return SL.LogReg(np.array(d["mu"]), np.array(d["sd"]), np.array(d["w"]), float(d["b"]), tuple(d["features"]))


def make_rule(d: dict):
    ml = d.get("max_len")
    ml = None if ml is None or (isinstance(ml, float) and np.isnan(ml)) else int(ml)
    if d["score"] == "lattice":
        return SL.Lattice(theta=float(d["theta"]), lam=float(d["lam"]), k=int(d["k"]), lag=int(d["lag"]), max_len=ml)
    return SL.Acceptor(score=d["score"], theta=float(d["theta"]), lam=float(d["lam"]),
                       gamma=float(d.get("gamma", 1.0)), max_len=ml,
                       shifts=load_shifts() if d["score"] == "class" else None,
                       model=load_linear(float(d["lam"])) if d["score"] == "linear" else None)


# ---- decoding and scoring
def decode(rule, st) -> list:
    pr = prior_for(st["row"])
    em = rule.decode(st["segs"], pr) if isinstance(rule, SL.Lattice) else FU.decode_fused(st["segs"], pr, rule)
    return S.collapse_repeats(em) if DS["collapse"] else em


def score(rule, sts, per_stream: bool = False):
    """Aggregate GER parts + noise false-accept rate (+ one row per stream)."""
    sc, rows, nb, na = [], [], 0, 0
    for st in sts:
        em = decode(rule, st)
        s = M.score_sequence(em, st["labels"], st["seg"], st["kinds"])
        acc = sum(FU.noise_false_accepts(em, [(a, b)], DS["noise"]["slack"])["blocks_accepted"]
                  for a, b, _ in st["blocks"])
        nb, na = nb + len(st["blocks"]), na + acc
        sc.append(s)
        if per_stream:
            rows.append({"row": st["row"], "participant": int(PARTICIPANT[st["row"]]), "n_ref": s["n_ref"],
                         "sub": s["sub"], "del": s["del"], "ins": s["ins"], "exact": s["exact"],
                         "blocks": len(st["blocks"]), "blocks_accepted": acc})
    agg = M.aggregate(sc)
    out = {k: agg[k] for k in ("ger", "sub_rate", "del_rate", "ins_rate", "sentence_acc", "latency_median")}
    out["noise_fa"] = na / nb if nb else float("nan")
    if isinstance(rule, SL.Lattice):
        out["decision_delay_median"] = float(np.median(np.concatenate(
            [SL.decision_delays(st["segs"], rule.lag) for st in sts if st["segs"]] or [np.zeros(1)])))
    return (out, pd.DataFrame(rows)) if per_stream else out


def sweep(name: str, rules: list) -> pd.DataFrame:
    """Score every rule on both selection variants; resumable per rule."""
    path = RES / f"sweep_{name}.json"
    done = json.loads(path.read_text()) if path.exists() else []
    have = {json.dumps(r["rule"], sort_keys=True) for r in done}
    todo = [r for r in rules if json.dumps(r.as_dict(), sort_keys=True) not in have]
    for rule in tqdm(todo, desc=f"sweep {name}"):
        row = {"arm": name, "rule": rule.as_dict()}
        for v in CONFIG["selection_variants"]:
            row[v] = score(rule, streams(v, "select"))
        done.append(row)
        if len(done) % 10 == 0 or rule is todo[-1]:
            write_json(path, done)
    return flat(done)


def flat(rows: list) -> pd.DataFrame:
    """Sweep rows -> one row per rule, metrics as `<metric>@<variant>` columns."""
    out = []
    for r in rows:
        d = {"arm": r["arm"], **r["rule"]}
        for v in CONFIG["selection_variants"]:
            for k, x in r[v].items():
                d[f"{k}@{v}"] = x
        out.append(d)
    return pd.DataFrame(out)


def group_of(r) -> str:
    """Rule family a θ sweep belongs to (every rule field except θ)."""
    if r["score"] == "lattice":
        return f"lattice lag {int(r['lag'])} k{int(r['k'])} λ{r['lam']:g}"
    return f"{r['score']} λ{r['lam']:g}" + (f" γ{r['gamma']:g}" if r["score"] == "class" else "")


def eligible(sw: pd.DataFrame, refs: dict) -> np.ndarray:
    """The budget (see the title cell): every limit met on the selection signers."""
    bud, ref = CONFIG["budget"], refs[CONFIG["budget"]["reference"]]
    ok = np.ones(len(sw), bool)
    for v in CONFIG["selection_variants"]:
        for m in ("sub_rate", "ins_rate"):
            ok &= (sw[f"{m}@{v}"] <= ref[v][m] + bud["tol_rate"]).to_numpy()
    return ok & (sw["noise_fa@sentence-noisy"] <= ref["sentence-noisy"]["noise_fa"] + bud["tol_noise_fa"]).to_numpy()


def all_sweeps() -> pd.DataFrame:
    """Every sweep file, one row per distinct rule, with its family."""
    sw = pd.concat([flat(json.loads(p.read_text())) for p in sorted(RES.glob("sweep_*.json"))], ignore_index=True)
    sw["group"] = sw.apply(group_of, axis=1)
    return sw.drop_duplicates(subset=[c for c in sw.columns if "@" not in c and c != "arm"])


print(f"forward cache: {FWD}\nout: {OUT}\nD3 nu={NU} min_len={MIN_LEN} · prior {ARM} [{FAMILY}]")
print(f"selection signers {SELECT} · {len(EVAL)} evaluation signers")

## 1. The floor, measured properly: segment labels and the budget

Each selection-signer D3 segment's `argmax q` is aligned against the reference with
every segment accepted: `match`, `sub` or `ins`, the true gloss, and its rank in `q`. The
three references are then decoded for real:
- `baseline`: no floor;
- `floor`: θ=0.3;
- `floor+prior`: rescore λ=0.3, θ=0.3.

The **floor's** selection-signer rates are the budget every arm must stay within. A parity
check makes sure `floor` reproduces the next-gloss sweep's `none θ=0.3` row.

In [ ]:
# ============================================================
# Segment labels (alignment) + reference decoders on the selection signers
# ============================================================
lab_path, ref_path = RES / "labels_select.parquet", RES / "refs_select.json"
if not lab_path.exists():
    rows = []
    for v in CONFIG["selection_variants"]:
        for st in streams(v, "select"):
            for seg, lab in zip(st["segs"], SL.label_segments(st["segs"], st["labels"])):
                q1 = float(seg.q.max())
                rank = int((seg.q > seg.q[lab["true"]]).sum()) if lab["true"] >= 0 else -1
                rows.append({"variant": v, "row": st["row"], "participant": int(PARTICIPANT[st["row"]]),
                             "start": seg.start, "len": seg.length, "q1": q1, **lab, "true_rank": rank})
    pd.DataFrame(rows).to_parquet(lab_path)
LAB = pd.read_parquet(lab_path)

if not ref_path.exists():
    write_json(ref_path, {name: {v: score(make_rule(d), streams(v, "select")) for v in CONFIG["selection_variants"]}
                          for name, d in CONFIG["references"].items()})
REFS = json.loads(ref_path.read_text())

if LIMIT_PER_GROUP is None:  # parity with the next-gloss sweep (same streams, same rule)
    ng = pd.DataFrame(json.loads((CACHE_DIR / CONFIG["next_gloss_results"] / "sweep_D3_none.json").read_text()))
    ng = ng[(ng.theta == 0.3) & ng.max_len.isna()].iloc[0]
    print(f"parity: floor GER {REFS['floor']['sentence-noisy']['ger']:.5f} vs next-gloss sweep {ng.ger:.5f}")
    assert abs(REFS["floor"]["sentence-noisy"]["ger"] - ng.ger) < 1e-9

THETA_REF = CONFIG["references"]["floor"]["theta"]
LAB["kept"] = LAB.q1 >= THETA_REF
print("segments by alignment op and floor decision (selection signers):")
print(pd.crosstab([LAB.variant, LAB.op], LAB.kept.map({True: "kept", False: "rejected"}), margins=True))
rej = LAB[~LAB.kept & (LAB.op == "sub")]
print("\nrejected substitutions: true gloss in q's top-k:",
      {k: f"{(rej.true_rank < k).mean():.0%}" for k in (2, 3, 5, 10)})
pd.DataFrame(REFS).T.map(lambda d: {k: round(v, 3) for k, v in d.items()
                                    if k in ("ger", "sub_rate", "del_rate", "ins_rate", "noise_fa")})

## 2. Arm `scores`: the same decision with another number

`q`, `peak`, `margin` and `qp`, each over θ, with no prior (λ=0) and with the trigram
(λ=0.3, the class picked from `q·p^λ`). `q` with λ=0 at θ=0.3 is the floor itself.

In [ ]:
# ============================================================
# Sweep: score x lam x theta
# ============================================================
A = CONFIG["arms"]["scores"]
rules = [SL.Acceptor(score=s, theta=t, lam=l) for s in A["score"] for l in A["lam"] for t in CONFIG["theta_grid"]]
SW_SCORES = sweep("scores", rules)
SW_SCORES.sort_values("del_rate@sentence-clean").groupby(["score", "lam"]).head(1).round(3)

## 3. Arm `class`: a floor per sign

`shift_c` is the log-odds by which segments voted as gloss `c` are right more (or less)
often than average on the selection signers (clean + noisy). It is shrunk toward 0 with
`alpha` pseudo-segments, so rare glosses keep the global floor. The score is
`sigmoid(logit(q_c) + γ·shift_c)`, compared with θ. That is a per-sign floor.

In [ ]:
# ============================================================
# Per-gloss shifts from the selection labels, then sweep gamma x lam x theta
# ============================================================
A = CONFIG["arms"]["class"]
sh_path = RES / "class_shifts.npy"
if not sh_path.exists():
    LAB = pd.read_parquet(RES / "labels_select.parquet")
    np.save(sh_path, SL.class_shifts(LAB["pred"].to_numpy(), (LAB["op"] == M.MATCH).to_numpy(), N_CLASSES, A["alpha"]))
SH = load_shifts()
print("shift range", SH.min().round(2), SH.max().round(2), "· glosses shifted by >0.5:", int((np.abs(SH) > 0.5).sum()))
rules = [SL.Acceptor(score="class", theta=t, lam=l, gamma=g, shifts=SH)
         for g in A["gamma"] for l in A["lam"] for t in CONFIG["theta_grid"]]
SW_CLASS = sweep("class", rules)
SW_CLASS.sort_values("del_rate@sentence-clean").groupby(["gamma", "lam"]).head(1).round(3)

## 4. Arm `lattice`: wait for the next sign, then decide

A segment is held until `lag` more segments have closed. The best path over every held
segment's top-k glosses or "skip" is then chosen, and only the oldest is committed.
Accepting gloss c scores `log q_c + λ·log(C·p(c | path)) − log θ`, and skipping scores 0.
`lag=0` is the same rule without waiting, which is the fair comparison for `lag` 1–2.
The cost is delay: `decision_delay_median` is the extra frames before an answer. The
emission keeps the sign's own frame.

**The longest cell.**

In [ ]:
# ============================================================
# Sweep: lam x k x lag x theta
# ============================================================
A = CONFIG["arms"]["lattice"]
rules = [SL.Lattice(theta=t, lam=l, k=k, lag=g) for g in A["lag"] for k in A["k"] for l in A["lam"] for t in A["theta"]]
SW_LAT = sweep("lattice", rules)
SW_LAT.sort_values("del_rate@sentence-clean").groupby(["lag", "k", "lam"]).head(1).round(3)

## 5. Arm `linear`: a learned accept/reject

**Fits a model** (a logistic regression, seconds on CPU). Every selection-signer segment
(clean + noisy) gets the 10 features in `sb.recognize.continuous.select.FEATURES` for the
gloss it would accept at that λ. Its label is whether that gloss is right. Noise and
spurious segments are always wrong. The L2 strength is chosen by leave-one-signer-out AUC
over the 5 selection signers. Then one model is fitted on all 5, its weights are saved as
plain numbers (what the client would load), and θ, the minimum predicted probability of
being right, is swept.

In [ ]:
# ============================================================
# Features -> leave-one-signer-out AUC per l2 -> final fit -> sweep theta
# ============================================================
A = CONFIG["arms"]["linear"]


def auc(y, s):
    r = pd.Series(s).rank().to_numpy()
    npos = y.sum()
    return (r[y == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(y) - npos))


def feature_table(lam: float):
    probe = SL.Acceptor(score="q", lam=lam)
    X, y, g = [], [], []
    for v in CONFIG["selection_variants"]:
        for st in streams(v, "select"):
            pr, hist = prior_for(st["row"]), ()
            for seg, lab in zip(st["segs"], SL.label_segments(st["segs"], st["labels"])):
                c, f, _ = probe.pick(seg, hist, pr)
                X.append(SL.features(seg, c, f, pr(hist), hist))
                y.append(float(lab["op"] in (M.MATCH, M.SUB) and lab["true"] == c))
                g.append(int(PARTICIPANT[st["row"]]))
                if not hist or hist[-1] != c:  # history: every segment accepted, repeats collapsed
                    hist = hist + (c,)
    return np.array(X), np.array(y), np.array(g)


LIN = {}
for lam in A["lam"]:
    path = RES / f"linear_lam{lam}.json"
    if not path.exists():
        X, y, g = feature_table(lam)
        cv = {l2: float(np.mean([auc(y[g == s], SL.LogReg.fit(X[g != s], y[g != s], l2).prob(X[g == s]))
                                 for s in np.unique(g)])) for l2 in A["l2"]}
        best = max(cv, key=cv.get)
        m = SL.LogReg.fit(X, y, best)
        write_json(path, {**m.to_dict(), "l2": best, "cv_auc": {str(k): v for k, v in cv.items()},
                          "cv_auc_best": cv[best], "n": int(len(y)), "positive_rate": float(y.mean()),
                          "auc_q_only": float(auc(y, X[:, SL.FEATURES.index("log_q")]))})
    LIN[lam] = json.loads(path.read_text())
    print(f"lam {lam}: l2 {LIN[lam]['l2']} · leave-one-signer-out AUC {LIN[lam]['cv_auc_best']:.3f}"
          f" (q alone, in-sample {LIN[lam]['auc_q_only']:.3f})")
    print("  weights (standardized):", dict(zip(LIN[lam]["features"], np.round(LIN[lam]["w"], 2))))

rules = [SL.Acceptor(score="linear", theta=t, lam=l, model=load_linear(l)) for l in A["lam"] for t in CONFIG["theta_grid"]]
SW_LIN = sweep("linear", rules)
SW_LIN.sort_values("del_rate@sentence-clean").groupby(["lam"]).head(1).round(3)

## 5b. Refine θ at each family's budget edge

The grid's θ steps (0.05, and 0.1 for the lattice above 0.3) are coarse next to the
budget's 0.002 tolerance. In the first run (2026-09-24), every family's best eligible grid
point sat well *inside* the budget, with more deletions than the floor. The next step
down broke a limit, almost always extra signs on noisy streams. For each family, this cell
sweeps `refine_steps` values between its lowest eligible θ and the grid point below it.
It then sweeps again inside the new bracket, so the edge is resolved to about 0.002.

In [ ]:
# ============================================================
# Two rounds of finer theta between each family's last eligible and first ineligible point
# ============================================================
N_STEPS = CONFIG["refine_steps"]
REFS = json.loads((RES / "refs_select.json").read_text())
for rnd in range(2):
    SW = all_sweeps()
    SW["eligible"] = eligible(SW, REFS)
    rules = []
    for g, d in SW.groupby("group"):
        d = d.sort_values("theta")
        ok = d[d.eligible]
        below = d[d.theta < ok.theta.min()] if len(ok) else d.iloc[:0]
        if ok.empty or below.empty:
            continue
        base = {k: d.iloc[0][k] for k in ("score", "lam", "gamma", "k", "lag", "max_len") if k in d}
        base = {k: v for k, v in base.items() if v is not None and not (isinstance(v, float) and np.isnan(v))}
        for t in np.linspace(below.theta.max(), ok.theta.min(), N_STEPS + 2)[1:-1]:
            rules.append(make_rule({**base, "theta": round(float(t), 4)}))
    sweep(f"refine{rnd}", rules)
SW = all_sweeps()
SW["eligible"] = eligible(SW, REFS)
print(f"{len(SW)} settings, {int(SW.eligible.sum())} eligible")

## 6. Budget-matched choice → evaluation signers, with a per-signer bootstrap

For each arm group, the eligible setting (§ title) with the lowest summed deletion rate on
the selection signers is kept. The chosen settings and the three references are then
decoded on the evaluation signers, all four variants. **Δ vs `floor`** comes with 95%
intervals from resampling the 16 evaluation signers. An arm **wins** if Δdeletions < 0,
with neither Δsubstitutions nor Δinsertions above 0, and the same for noise.

In [ ]:
# ============================================================
# Eligible settings on the selection signers -> chosen per group -> eval + bootstrap
# ============================================================
import hashlib

BUD = CONFIG["budget"]
REFS = json.loads((RES / "refs_select.json").read_text())
SW = all_sweeps()
SW["eligible"] = eligible(SW, REFS)
SW["del_sum"] = sum(SW[f"del_rate@{v}"] for v in CONFIG["selection_variants"])
SW["family"] = SW["group"].str.replace(r" k\d+ λ[\d.]+$", "", regex=True)  # lattice: best k, λ per lag
CH = SW[SW.eligible].sort_values("del_sum").groupby("family").head(1).sort_values("del_sum")
print(f"eligible settings: {int(SW.eligible.sum())} of {len(SW)}")
RULE_COLS = ["score", "theta", "lam", "gamma", "k", "lag", "max_len"]
CHOSEN = {"baseline": CONFIG["references"]["baseline"], "floor": CONFIG["references"]["floor"],
          "floor+prior": CONFIG["references"]["floor+prior"]}
for _, r in CH.iterrows():
    CHOSEN[r.family] = {k: r[k] for k in RULE_COLS
                       if k in r and r[k] is not None and not (isinstance(r[k], float) and np.isnan(r[k]))}
display(CH[["family", "theta", "lam", "k", "lag", "del_sum", "del_rate@sentence-clean", "sub_rate@sentence-clean",
            "ins_rate@sentence-clean", "del_rate@sentence-noisy", "noise_fa@sentence-noisy"]].round(4))

VARIANTS = ["sentence-clean", "sentence-noisy", "control-clean", "control-noisy"]
final_rows = []
for label, d in tqdm(CHOSEN.items(), desc="evaluation signers"):
    for v in VARIANTS:
        key = hashlib.sha1(json.dumps(d, sort_keys=True, default=float).encode()).hexdigest()[:10]
        path = PER_STREAM / f"{key}__{v}.parquet"
        jpath = path.with_suffix(".json")
        if not path.exists():
            agg, per = score(make_rule(d), streams(v, "eval"), per_stream=True)
            write_json(jpath, agg)
            per.to_parquet(path)
        final_rows.append({"label": label, "variant": v, "key": key, **{k: d.get(k) for k in RULE_COLS}, **json.loads(jpath.read_text())})
FINAL = pd.DataFrame(final_rows)


KEYS = FINAL.drop_duplicates("label").set_index("label")["key"]


def boot(label: str, v: str) -> dict:
    a = pd.read_parquet(PER_STREAM / f"{KEYS[label]}__{v}.parquet").groupby("participant").sum()
    b = pd.read_parquet(PER_STREAM / f"{KEYS['floor']}__{v}.parquet").groupby("participant").sum()
    rng = np.random.default_rng(CONFIG["bootstrap"]["seed"])
    idx = rng.integers(0, len(a), (CONFIG["bootstrap"]["n"], len(a)))
    out = {"label": label, "variant": v}
    for m in ("sub", "del", "ins"):
        d = a[m].to_numpy()[idx].sum(1) / a["n_ref"].to_numpy()[idx].sum(1) - \
            b[m].to_numpy()[idx].sum(1) / b["n_ref"].to_numpy()[idx].sum(1)
        pt = a[m].sum() / a["n_ref"].sum() - b[m].sum() / b["n_ref"].sum()
        out[f"Δ{m}"], out[f"Δ{m}_lo"], out[f"Δ{m}_hi"] = pt, *np.percentile(d, [2.5, 97.5])
    if a["blocks"].sum():
        d = a["blocks_accepted"].to_numpy()[idx].sum(1) / a["blocks"].to_numpy()[idx].sum(1) - \
            b["blocks_accepted"].to_numpy()[idx].sum(1) / b["blocks"].to_numpy()[idx].sum(1)
        out["Δnoise_fa"] = a["blocks_accepted"].sum() / a["blocks"].sum() - b["blocks_accepted"].sum() / b["blocks"].sum()
        out["Δnoise_fa_lo"], out["Δnoise_fa_hi"] = np.percentile(d, [2.5, 97.5])
    return out


DELTAS = pd.DataFrame([boot(l, v) for l in CHOSEN if l != "floor" for v in VARIANTS])
DELTAS["wins"] = (DELTAS["Δdel_hi"] < 0) & (DELTAS["Δsub_lo"] <= 0) & (DELTAS["Δins_lo"] <= 0) & \
                 (DELTAS["Δsub"] <= BUD["tol_rate"]) & (DELTAS["Δins"] <= BUD["tol_rate"])
DELTAS.to_csv(RES / "deltas.csv", index=False)
TABLE = FINAL.pivot_table(index="label", columns="variant", values=["ger", "sub_rate", "del_rate", "ins_rate"])
TABLE.to_csv(RES / "final_table.csv")
write_json(RES / "final.json", {"chosen": CHOSEN, "results": FINAL.to_dict(orient="records"),
                                "deltas": DELTAS.to_dict(orient="records"), "budget": BUD,
                                "select_signers": SELECT, "eval_signers": EVAL})
print(FINAL[FINAL.variant.str.startswith("sentence")].pivot_table(
    index="label", columns="variant", values=["ger", "sub_rate", "del_rate", "ins_rate", "noise_fa"]).round(3).to_string())
DELTAS[DELTAS.variant.str.startswith("sentence")].round(4)

## 7. Figures

In [ ]:
# ============================================================
# Selection trade-off: missed vs wrong+extra along theta, per chosen group; eval breakdown
# ============================================================
SW = all_sweeps()
REFS = json.loads((RES / "refs_select.json").read_text())
FIN = json.loads((RES / "final.json").read_text())
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, v in zip(axes, CONFIG["selection_variants"]):
    for label, d in FIN["chosen"].items():
        if label in ("baseline", "floor", "floor+prior"):
            continue
        m = np.ones(len(SW), bool)
        for k, x in d.items():
            if k != "theta" and k in SW:
                m &= (SW[k] == x).to_numpy()
        g = SW[m].sort_values("theta")
        ax.plot(g[f"sub_rate@{v}"] + g[f"ins_rate@{v}"], g[f"del_rate@{v}"], ".-", label=label, alpha=0.8)
    for name, mk in (("floor", "k*"), ("floor+prior", "r*"), ("baseline", "ko")):
        r = REFS[name][v]
        ax.plot(r["sub_rate"] + r["ins_rate"], r["del_rate"], mk, ms=12, label=name)
    ax.set_xlabel("wrong + extra signs per signed sign")
    ax.set_ylabel("missed signs per signed sign")
    ax.set_title(f"selection signers, {v} (down-left is better)")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(ASSETS / "tradeoff_select.png", dpi=130)
plt.show()

F = pd.DataFrame(FIN["results"])
F = F[F.variant.isin(["sentence-clean", "sentence-noisy"])].set_index(["variant", "label"])[["sub_rate", "del_rate", "ins_rate"]]
ax = F.plot.barh(stacked=True, figsize=(9, 0.35 * len(F) + 2))
ax.invert_yaxis()
ax.set_xlabel("GER parts (evaluation signers)")
plt.tight_layout()
plt.savefig(ASSETS / "breakdown_eval.png", dpi=130)
plt.show()

## 8. Reading the results (Claude fills this into `docs/reports/sign-to-speech-downstream.md` §2.2)

- Which arms are **eligible** at the floor's budget, and how many missed signs does each
  save on the evaluation signers (Δdel with its interval)?
- Does anything that works on clean sentences survive noise? `peak` did not in the
  diagnostic.
- `lattice`: does waiting help (`lag` 1–2 vs 0), and how many frames of delay does it cost?
- `linear`: which features carry weight, and does it beat `q` on held-out signers (AUC)?
- Is `floor+prior` alone already a partial answer? It had fewer deletions than the floor
  in the next-gloss run.
- Does the winner hold on `control` (random sequences), where the prior has nothing to offer?
- Should the retrain with noise as null (phase 3) go ahead?